In [ ]:
# ==============================================================
#  CNN–Siamese Fingerprint Recognition (Auto Kaggle Upload Version)
#  - Prompts to upload kaggle.json interactively
#  - Downloads SOCOFing dataset
#  - Builds & trains CNN–Siamese (ResNet18 + SE attention)
#  - Evaluates ROC/EER/AUC + FAR/FRR + Acc/Prec/Rec/F1 + Confusion Matrix
#  - Reports tiny 1:N Identify metrics (Top-1 / Top-5)
#  - Performs Verify (1:1) and Identify (1:N)
# ==============================================================

# ---------------- Kaggle Authentication ----------------
import os, shutil
from google.colab import files
from pathlib import Path

print("📁 Please upload your kaggle.json file (downloaded from Kaggle > Account > API):")
uploaded = files.upload()  # interactive upload
if 'kaggle.json' not in uploaded:
    raise FileNotFoundError("❌ kaggle.json not found. Please upload when prompted.")

os.makedirs("/root/.kaggle", exist_ok=True)
shutil.move("kaggle.json", "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 0o600)
print("✅ Kaggle API key installed successfully!\n")
### Note: If you run this code outside of Colab, ensure you have your kaggle.json in the correct location (~/.kaggle/kaggle.json) with proper permissions (600) for the Kaggle API to work.
# ---------------- Install Dependencies ----------------
!pip -q install kaggle faiss-cpu torch torchvision scikit-learn

# ---------------- Imports ----------------
import re, random, zipfile
from typing import List, Dict
import numpy as np
from PIL import Image

import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from torchvision import models
from sklearn.metrics import roc_curve, auc, confusion_matrix, precision_recall_fscore_support

try:
    import faiss
except Exception:
    faiss = None

# ---------------- Project Paths ----------------
BASE = Path("/content/fingerprint_project")
DATA = BASE/"data"
ART  = BASE/"artifacts"
for p in [DATA, ART]: p.mkdir(parents=True, exist_ok=True)

# ---------------- Download Dataset (SOCOFing) ----------------
print("[Downloading SOCOFing dataset from Kaggle...]")
!kaggle datasets download -d ruizgara/socofing -p /content
!unzip -qo /content/socofing.zip -d /content
print("✅ SOCOFing dataset ready.\n")

# ---------------- Prepare train/val splits ----------------
def clean_dir(p: Path):
    if p.exists(): shutil.rmtree(p)
    p.mkdir(parents=True, exist_ok=True)

root_train = DATA/"prepared/train"
root_val   = DATA/"prepared/val"
clean_dir(root_train); clean_dir(root_val)

real_dirs = list((Path("/content/SOCOFing")).glob("**/Real"))
if not real_dirs: real_dirs = [Path("/content/SOCOFing/Real")]
files = []
for rd in real_dirs:
    files.extend([p for p in rd.rglob("*") if p.suffix.lower() in {".bmp",".png",".jpg",".jpeg"}])

def pid_from_name(fp: Path):
    m = re.match(r"(\d+)__", fp.name)
    return m.group(1) if m else fp.stem.split("__")[0]

id2files: Dict[str, List[Path]] = {}
for fp in files:
    pid = pid_from_name(fp)
    id2files.setdefault(pid, []).append(fp)

ids = [i for i in id2files.keys() if len(id2files[i]) >= 4]
random.seed(42); random.shuffle(ids)
split = max(1, int(0.8*len(ids)))
train_ids, val_ids = ids[:split], ids[split:]

def copy_group(id_list, out_root):
    for pid in id_list:
        out = out_root/pid
        out.mkdir(parents=True, exist_ok=True)
        for src in id2files[pid]:
            Image.open(src).convert("L").save(out/f"{src.stem}.png")

copy_group(train_ids, root_train)
copy_group(val_ids, root_val)
print(f"✅ Prepared IDs — train: {len(train_ids)}, val: {len(val_ids)}\n")

# ---------------- Dataset Class ----------------
def list_people_folders(root: Path) -> List[Path]:
    return [root/p for p in sorted(os.listdir(root)) if (root/p).is_dir()]

def list_images(root: Path) -> List[Path]:
    exts = {".png",".bmp",".jpg",".jpeg",".tif",".tiff"}
    return [root/f for f in sorted(os.listdir(root)) if (root/f).suffix.lower() in exts]

class PairDataset(Dataset):
    """Random genuine/impostor pairs for Siamese training."""
    def __init__(self, root: Path, transform=None):
        self.root = root; self.transform = transform
        self.people = list_people_folders(root)
        self.per_id_imgs = {p.name: list_images(p) for p in self.people}
        self.flat = [(p.name, img) for p in self.people for img in self.per_id_imgs[p.name]]
    def __len__(self): return len(self.flat)
    def __getitem__(self, idx):
        pid, img_path = self.flat[idx]
        same = random.randint(0, 1) == 1
        pid2 = pid if same else random.choice([x.name for x in self.people if x.name != pid])
        img1 = Image.open(img_path).convert("L")
        img2 = Image.open(random.choice(self.per_id_imgs[pid2])).convert("L")
        if self.transform:
            img1,img2 = self.transform(img1),self.transform(img2)
        label = torch.tensor([1.0 if same else 0.0],dtype=torch.float32)
        return img1,img2,label

# ---------------- Transforms ----------------
IMG_SIZE = (224,224)
train_tf = T.Compose([
    T.Resize(IMG_SIZE),
    T.RandomRotation(12),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize([0.5],[0.5]),
])
eval_tf = T.Compose([
    T.Resize(IMG_SIZE),
    T.ToTensor(),
    T.Normalize([0.5],[0.5]),
])

# ---------------- Model ----------------
EMBED_DIM = 128
class SEBlock(nn.Module):
    def __init__(self,c,r=8):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(c,c//r),nn.ReLU(inplace=True),
            nn.Linear(c//r,c),nn.Sigmoid())
    def forward(self,x):
        w = x.mean((2,3)); s = self.fc(w)
        return x * s.unsqueeze(-1).unsqueeze(-1)

class CNNEncoder(nn.Module):
    def __init__(self,embed_dim=EMBED_DIM):
        super().__init__()
        base = models.resnet18(weights=None)
        base.conv1 = nn.Conv2d(1,64,7,2,3,bias=False)
        self.backbone = nn.Sequential(*list(base.children())[:-2])
        self.se = SEBlock(512)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512,256),nn.ReLU(inplace=True),
            nn.Linear(256,embed_dim))
    def forward(self,x):
        f = self.backbone(x)
        f = self.se(f)
        f = self.pool(f)
        z = self.head(f)
        return nn.functional.normalize(z,dim=1)

class SiameseNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = CNNEncoder()
        self.dist = nn.PairwiseDistance(p=2)
    def forward(self,x1,x2):
        e1=self.encoder(x1); e2=self.encoder(x2)
        d=self.dist(e1,e2)
        return e1,e2,d

class ContrastiveLoss(nn.Module):
    def __init__(self,margin=1.0): super().__init__(); self.margin=margin
    def forward(self,dist,label):
        pos=label.squeeze()*(dist**2)
        neg=(1-label.squeeze())*torch.clamp(self.margin-dist,min=0.0)**2
        return (pos+neg).mean()

# ---------------- Training ----------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE, EPOCHS, LR = 32, 6, 1e-4

def make_loaders():
    tr = PairDataset(root_train, transform=train_tf)
    va = PairDataset(root_val,   transform=eval_tf)
    return (DataLoader(tr, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True),
            DataLoader(va, batch_size=BATCH_SIZE, shuffle=False, num_workers=2))

def eval_epoch(model, val_loader, criterion):
    model.eval(); loss=0
    with torch.no_grad():
        for x1,x2,y in val_loader:
            x1,x2,y = x1.to(DEVICE),x2.to(DEVICE),y.to(DEVICE)
            _,_,d=model(x1,x2); loss += criterion(d,y).item()
    return loss/len(val_loader)

def train():
    tr,va = make_loaders()
    model = SiameseNet().to(DEVICE)
    crit  = ContrastiveLoss()
    opt   = optim.Adam(model.parameters(), lr=LR)
    best  = float("inf")
    best_path = ART/"siamese_best.pt"
    for ep in range(1,EPOCHS+1):
        model.train(); tr_loss=0
        for x1,x2,y in tr:
            x1,x2,y=x1.to(DEVICE),x2.to(DEVICE),y.to(DEVICE)
            opt.zero_grad();_,_,d=model(x1,x2)
            loss=crit(d,y); loss.backward(); opt.step(); tr_loss+=loss.item()
        tr_loss/=len(tr)
        va_loss=eval_epoch(model,va,crit)
        print(f"Epoch {ep}/{EPOCHS} | train {tr_loss:.4f} | val {va_loss:.4f}")
        if va_loss<best:
            best=va_loss; torch.save(model.state_dict(), best_path)
            print("  ↳ saved best.pt")
    return model, best_path

# ---------------- Metrics helpers ----------------
def compute_eer(y_true,y_score):
    fpr,tpr,thr=roc_curve(y_true,y_score,pos_label=1)
    fnr=1-tpr; idx=int(np.argmin(np.abs(fnr-fpr)))
    return (fnr[idx]+fpr[idx])/2, float(thr[idx]), fpr, tpr, thr

def threshold_metrics(y_true, y_score, tau):
    """
    Report FAR (FPR), FRR (FNR), Accuracy, Precision, Recall, F1, Confusion Matrix @ tau.
    y_true: 1=genuine, 0=impostor; y_score=similarity (higher=more similar).
    """
    # convert similarity → predicted label at tau using the same decision rule used in verify()
    y_pred = (y_score >= tau).astype(np.int32)  # similar enough → genuine
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    acc = (tp + tn) / max(1, (tp+tn+fp+fn))
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
    far = fp / max(1, (fp+tn))  # impostors accepted (FPR)
    frr = fn / max(1, (tp+fn))  # genuines rejected (FNR)
    return {
        "tau": float(tau),
        "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn),
        "Accuracy": float(acc),
        "Precision": float(prec),
        "Recall": float(rec),
        "F1": float(f1),
        "FAR": float(far),
        "FRR": float(frr),
    }

# ---------------- Evaluate (adds full metrics) ----------------
# ---------------- Enhanced Evaluate (adds full metrics + formatted table) ----------------
def evaluate(model, weights_path):
    _, va = make_loaders()
    model.load_state_dict(torch.load(weights_path, map_location=DEVICE))
    model.eval()
    labels, scores = [], []

    with torch.no_grad():
        for x1, x2, y in va:
            x1, x2 = x1.to(DEVICE), x2.to(DEVICE)
            _, _, d = model(x1, x2)
            sim = 1 / (1 + d.cpu().numpy())  # similarity (higher=better)
            scores += sim.tolist()
            labels += y.numpy().tolist()

    y_true = np.array(labels).ravel().astype(np.int32)
    y_score = np.array(scores).ravel().astype(np.float32)

    # --- Compute EER, AUC, etc. ---
    eer, tau, fpr, tpr, thr = compute_eer(y_true, y_score)
    auc_val = auc(fpr, tpr)
    tm = threshold_metrics(y_true, y_score, tau)

    # --- Compute Accuracy, Precision, Recall, F1 manually ---
    acc = tm["Accuracy"]
    prec = tm["Precision"]
    rec = tm["Recall"]
    f1 = tm["F1"]
    far = tm["FAR"]
    frr = tm["FRR"]

    print("\n=== Verification Metrics (Validation Pairs) ===")
    print(f"Dataset Used: SOCOFing")
    print(f"EER: {eer*100:.2f}%")
    print(f"AUC: {auc_val:.4f}")
    print(f"Threshold (τ): {tau:.4f}")
    print(f"FAR@τ: {far*100:.2f}%   FRR@τ: {frr*100:.2f}%")
    print(f"Accuracy@τ: {acc*100:.2f}%  Precision@τ: {prec*100:.2f}%  Recall@τ: {rec*100:.2f}%  F1@τ: {f1*100:.2f}%")
    print(f"Confusion Matrix@τ → TP:{tm['TP']}  FP:{tm['FP']}  TN:{tm['TN']}  FN:{tm['FN']}")

    # --- Print table-style summary for documentation ---
    print("\n" + "="*95)
    print(f"{'Dataset Used':<15} {'Accuracy (%)':<15} {'Precision (%)':<15} {'Recall (%)':<12} "
          f"{'F1-Score (%)':<15} {'AUC':<10} {'EER (%)':<10}")
    print("-"*95)
    print(f"{'SOCOFing':<15} {acc*100:<15.2f} {prec*100:<15.2f} {rec*100:<12.2f} "
          f"{f1*100:<15.2f} {auc_val:<10.4f} {eer*100:<10.2f}")
    print("="*95)

    return float(tau)


# ---------------- Tiny 1:N identification metrics (Top-k) ----------------
def build_gallery(root: Path, max_ids=12, max_imgs=2):
    gal = BASE/"gallery"
    clean_dir(gal)
    ids = [d for d in root.iterdir() if d.is_dir()]
    random.shuffle(ids); ids = ids[:max_ids]
    for pid in ids:
        out = gal/pid.name; out.mkdir(parents=True, exist_ok=True)
        pics = list(pid.glob("*.png"))
        random.shuffle(pics)
        for fp in pics[:max_imgs]:
            Image.open(fp).convert("L").save(out/fp.name)
    return gal

def embed_one(model, fp: Path):
    with torch.no_grad():
        x = eval_tf(Image.open(fp).convert("L")).unsqueeze(0).to(DEVICE)
        z = model.encoder(x).cpu().numpy()[0].astype("float32")
    return z

def identify_topk_metrics(model, gallery_dir: Path, topk_list=(1,5), probes_per_id=1):
    # Build gallery embeddings & labels
    labels, embs, id_of = [], [], []
    for pdir in sorted([d for d in gallery_dir.iterdir() if d.is_dir()]):
        for fp in sorted(pdir.glob("*.png")):
            labels.append(f"{pdir.name}/{fp.name}")
            id_of.append(pdir.name)
            embs.append(embed_one(model, fp))
    if not embs:
        return {k: 0.0 for k in topk_list}
    embs = np.vstack(embs).astype("float32")
    ids  = np.array(id_of)

    # Simple retrieval per-probe: pick extra images from each id as probes (or reuse one)
    hits = {k:0 for k in topk_list}; total = 0
    for pid in sorted(set(ids)):
        cand = [Path(gallery_dir/pid)/f.name for f in (gallery_dir/pid).glob("*.png")]
        if not cand: continue
        random.shuffle(cand)
        for probe in cand[:probes_per_id]:
            zq = embed_one(model, probe)[None, :]
            # L2 search (FAISS if available)
            if faiss is not None:
                index = faiss.IndexFlatL2(embs.shape[1]); index.add(embs)
                D, I = index.search(zq, max(topk_list))
                ranked = [labels[i] for i in I[0]]
            else:
                diffs = embs - zq[0]
                dists = np.sum(diffs*diffs, axis=1)
                idxs  = np.argsort(dists)[:max(topk_list)]
                ranked = [labels[i] for i in idxs]
            total += 1
            for k in topk_list:
                if any(r.startswith(f"{pid}/") for r in ranked[:k]):
                    hits[k] += 1
    return {k: (hits[k]/total if total>0 else 0.0) for k in topk_list}

# ---------------- Train & Evaluate ----------------
model, weights = train()
tau = evaluate(model, weights)

# ---------------- Verify & Identify ----------------
def verify(img1,img2,tau=None):
    tau=float(tau) if tau is not None else float(tau)
    tf=eval_tf
    x1=tf(Image.open(img1).convert("L")).unsqueeze(0).to(DEVICE)
    x2=tf(Image.open(img2).convert("L")).unsqueeze(0).to(DEVICE)
    with torch.no_grad():_,_,d=model(x1,x2)
    dist=float(d.item())
    print(f"[VERIFY] {img1.name} vs {img2.name} → dist={dist:.4f}, τ={tau:.4f} →",
          "MATCH" if dist<=tau else "NON-MATCH")

# Quick verify demo
val_ids=[d for d in root_val.iterdir() if d.is_dir()]
if len(val_ids)>=2:
    a,b=random.sample(val_ids,2)
    imgs_a=list(a.glob("*.png")); imgs_b=list(b.glob("*.png"))
    if len(imgs_a)>=2 and imgs_b:
        verify(imgs_a[0],imgs_a[1],tau)
        verify(imgs_a[0],imgs_b[0],tau)

# Tiny identify metrics on a mini gallery
gallery_dir = build_gallery(root_val, max_ids=12, max_imgs=2)
id_metrics = identify_topk_metrics(model, gallery_dir, topk_list=(1,5), probes_per_id=1)
print("\n=== Identification Metrics (Tiny Gallery) ===")
print(f"Top-1: {id_metrics[1]*100:.2f}%   Top-5: {id_metrics[5]*100:.2f}%")

print("\n✅ Completed full pipeline — with detailed metrics.")


📁 Please upload your kaggle.json file (downloaded from Kaggle > Account > API):


Saving kaggle.json to kaggle.json
✅ Kaggle API key installed successfully!

[Downloading SOCOFing dataset from Kaggle...]
Dataset URL: https://www.kaggle.com/datasets/ruizgara/socofing
License(s): other
socofing.zip: Skipping, found more recently modified local copy (use --force to force download)
✅ SOCOFing dataset ready.

✅ Prepared IDs — train: 480, val: 120

Epoch 1/6 | train 0.2108 | val 0.1828
  ↳ saved best.pt
Epoch 2/6 | train 0.1953 | val 0.1801
  ↳ saved best.pt
Epoch 3/6 | train 0.1859 | val 0.1779
  ↳ saved best.pt
Epoch 4/6 | train 0.1797 | val 0.1687
  ↳ saved best.pt
Epoch 5/6 | train 0.1796 | val 0.1654
  ↳ saved best.pt
Epoch 6/6 | train 0.1779 | val 0.1647
  ↳ saved best.pt

=== Verification Metrics (Validation Pairs) ===
Dataset Used: SOCOFing
EER: 23.25%
AUC: 0.8495
Threshold (τ): 0.6732
FAR@τ: 23.29%   FRR@τ: 23.21%
Accuracy@τ: 76.75%  Precision@τ: 75.89%  Recall@τ: 76.79%  F1@τ: 76.34%
Confusion Matrix@τ → TP:450  FP:143  TN:471  FN:136

Dataset Used    Accuracy (

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"GPU is available: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU available. Using CPU.")

# You can also use a system command to check NVIDIA GPU details
!nvidia-smi

When you run the `!nvidia-smi` command, you'll see details about the GPU, including its model name (e.g., Tesla T4), memory usage, and current processes. If you see 'Tesla T4' there, it confirms your session is running on that specific GPU.